In [ ]:
!pip install transformers accelerate bitsandbytes

In [3]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

In [6]:
# configure 4-bit quantization; weights are dequantized to float16 during forward pass
bnb_config = BitsAndBytesConfig(
     load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [9]:
def load_model(model_name):
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      quantization_config = bnb_config,
      device_map = "auto"
  )
  return model, tokenizer

base_model, base_tok = load_model("Qwen/Qwen2.5-0.5B")
instruct_model, instruct_tok = load_model("Qwen/Qwen2.5-0.5B-Instruct")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
prompt = "Q: What is 17 + 25?\nA:"
print(repr(prompt))  # should look exactly like your input, nothing added

inputs = base_tok(prompt, return_tensors="pt")
print(inputs["input_ids"])
print(base_tok.convert_ids_to_tokens(inputs["input_ids"][0]))

'Q: What is 17 + 25?\nA:'
tensor([[  48,   25, 3555,  374,  220,   16,   22,  488,  220,   17,   20, 5267,
           32,   25]])
['Q', ':', 'ĠWhat', 'Ġis', 'Ġ', '1', '7', 'Ġ+', 'Ġ', '2', '5', '?Ċ', 'A', ':']


Quick Test to see if the chat template is being applied to the instruct model and whether the base model receives plain strings (cells above and below)

In [11]:
messages = [{"role": "user", "content": "What is 17 + 25?"}]
formatted = instruct_tok.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(formatted)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 17 + 25?<|im_end|>
<|im_start|>assistant



In [13]:
def generate(model, tokenizer, prompt, max_new_tokens=200):
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=False
  )

  return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [16]:
import json

with open("/content/prompts.json") as f:
    harness = json.load(f)

results = []

for task in harness["prompts"]:
    task_id = task["id"]
    zero_shot_prompt = task["prompt"]
    few_shot_prompt = task["few_shot_prefix"] + task["prompt"]
    gold = task["gold_answer"]

    # --- base model, raw strings ---
    base_zero = generate(base_model, base_tok, zero_shot_prompt)
    base_few = generate(base_model, base_tok, few_shot_prompt)

    # --- instruct model, chat-templated ---
    def instruct_generate(text):
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": text}
        ]
        formatted = instruct_tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return generate(instruct_model, instruct_tok, formatted)

    instruct_zero = instruct_generate(zero_shot_prompt)
    instruct_few = instruct_generate(few_shot_prompt)

    results.append({
        "task_id": task_id,
        "category": task["category"],
        "difficulty": task["difficulty"],
        "gold_answer": gold,
        "base_zero_shot": base_zero,
        "base_few_shot": base_few,
        "instruct_zero_shot": instruct_zero,
        "instruct_few_shot": instruct_few,
    })

with open("/content/base_vs_instruct.json", "w") as f:
    json.dump(results, f, indent=2)